# Klasifikasi Bangun Ruang - 6 Kelas (FIXED)
## Perbaikan:
1. **Edge Detection (Canny + Sobel)** — Model fokus pada garis, tepi, bentuk geometris
2. **Preprocessing 3-channel**: Grayscale + Canny Edge + Sobel Edge
3. **Fine-tuning 2 fase** — Base model ikut belajar fitur bentuk
4. **Classifier head lebih kuat** — 2 hidden layer
5. **Augmentasi lebih lengkap**

In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

print("Library berhasil dimuat")
print(f"TensorFlow version: {tf.__version__}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Preprocessing: Edge Detection
Menggunakan 3 channel khusus agar model fokus pada **bentuk dan garis**:
- **Channel 1**: Grayscale (tekstur & intensitas)
- **Channel 2**: Canny Edge (garis tepi tajam)
- **Channel 3**: Sobel Magnitude (gradien tepi halus)

Dengan cara ini, informasi warna dihilangkan dan diganti dengan informasi **bentuk geometris**.

In [ ]:
def edge_detection_preprocessing(img):
    """
    Preprocessing fokus bentuk & garis:
    - Channel 1: Grayscale (tekstur)
    - Channel 2: Canny Edge Detection (garis tepi tajam)
    - Channel 3: Sobel Magnitude (gradien tepi halus)
    """
    # Pastikan uint8 untuk Canny
    img_uint8 = img.astype(np.uint8)

    # Channel 1: Grayscale
    gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)

    # Channel 2: Canny Edge Detection
    edges = cv2.Canny(img_uint8, threshold1=50, threshold2=150)

    # Channel 3: Sobel (gradien magnitude)
    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = np.sqrt(sobel_x**2 + sobel_y**2)
    sobel_mag = np.clip(sobel_mag / sobel_mag.max() * 255, 0, 255).astype(np.uint8)

    # Gabungkan jadi 3 channel
    result = np.stack([gray, edges, sobel_mag], axis=-1)

    # Normalisasi ke [-1, 1] (sesuai range MobileNetV2)
    result = (result.astype(np.float32) / 127.5) - 1.0

    return result

print("Fungsi edge detection preprocessing siap")

In [ ]:
# Visualisasi preprocessing untuk memastikan edge detection bekerja
import glob

dataset_path = '/content/drive/MyDrive/Skripsi Nila/Dataset'

# Ambil 1 gambar contoh dari setiap kelas
class_names = sorted(os.listdir(dataset_path))
fig, axes = plt.subplots(len(class_names), 4, figsize=(16, 4*len(class_names)))

for i, cls in enumerate(class_names):
    cls_path = os.path.join(dataset_path, cls)
    img_files = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg','.jpeg','.png','.jfif'))]
    if len(img_files) == 0:
        continue
    img_path = os.path.join(cls_path, img_files[0])
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))

    processed = edge_detection_preprocessing(img)
    # Denormalisasi untuk visualisasi
    vis = ((processed + 1.0) * 127.5).astype(np.uint8)

    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f'{cls} - Original')
    axes[i, 1].imshow(vis[:,:,0], cmap='gray')
    axes[i, 1].set_title(f'{cls} - Grayscale')
    axes[i, 2].imshow(vis[:,:,1], cmap='gray')
    axes[i, 2].set_title(f'{cls} - Canny Edge')
    axes[i, 3].imshow(vis[:,:,2], cmap='gray')
    axes[i, 3].set_title(f'{cls} - Sobel')

    for ax in axes[i]:
        ax.axis('off')

plt.suptitle('Visualisasi Edge Detection Preprocessing', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Data Generator - TANPA rescale (normalisasi sudah di preprocessing_function)
datagen = ImageDataGenerator(
    # TIDAK pakai rescale, normalisasi dilakukan di edge_detection_preprocessing
    preprocessing_function=edge_detection_preprocessing,

    # Augmentasi lebih lengkap untuk variasi bentuk
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    shear_range=0.15,

    validation_split=0.2
)

In [ ]:
train_generator = datagen.flow_from_directory(
    dataset_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_generator = datagen.flow_from_directory(
    dataset_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

print("\nKelas yang ditemukan:")
print(train_generator.class_indices)
print(f"\nJumlah data training: {train_generator.samples}")
print(f"Jumlah data validasi: {val_generator.samples}")
print(f"Jumlah kelas: {train_generator.num_classes}")

## Arsitektur Model
MobileNetV2 + Classifier Head yang lebih kuat:
- GlobalAveragePooling2D → Dense(256) → BatchNorm → Dropout(0.3) → Dense(128) → BatchNorm → Dropout(0.3) → Softmax

### Training 2 Fase:
1. **Fase 1**: Freeze semua base model, train head saja (10 epoch, LR=1e-3)
2. **Fase 2**: Unfreeze 30 layer terakhir, fine-tune (50 epoch, LR=1e-5)

In [ ]:
# Base Model
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Classifier Head yang lebih kuat
x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

x = Dense(128, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)

predictions = Dense(
    train_generator.num_classes,
    activation='softmax'
)(x)

model = Model(
    inputs=base_model.input,
    outputs=predictions
)

print(f"Total layers: {len(model.layers)}")
print(f"Base model layers: {len(base_model.layers)}")
model.summary()

## Fase 1: Train Classifier Head
Freeze semua layer base model, hanya train head baru (Dense layers).

In [ ]:
# ============ FASE 1: Train Head Only ============
# Freeze semua layer base model
for layer in base_model.layers:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Fase 1: Training classifier head...")
print(f"Trainable parameters: {sum(p.numpy().size for p in model.trainable_weights):,}")
print(f"Non-trainable parameters: {sum(p.numpy().size for p in model.non_trainable_weights):,}")

history_phase1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    verbose=1
)

print("\nFase 1 selesai!")
print(f"Training Accuracy: {history_phase1.history['accuracy'][-1]:.4f}")
print(f"Validation Accuracy: {history_phase1.history['val_accuracy'][-1]:.4f}")

## Fase 2: Fine-Tune Base Model
Unfreeze 30 layer terakhir MobileNetV2 agar model bisa belajar fitur **garis dan bentuk** yang spesifik untuk bangun ruang. Learning rate sangat kecil (1e-5) agar tidak merusak fitur yang sudah dipelajari.

In [ ]:
# ============ FASE 2: Fine-Tune Base Model ============
# Unfreeze 30 layer terakhir base model
for layer in base_model.layers[:-30]:
    layer.trainable = False
for layer in base_model.layers[-30:]:
    layer.trainable = True

# Compile ulang dengan learning rate kecil
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Fase 2: Fine-tuning base model...")
print(f"Trainable parameters: {sum(p.numpy().size for p in model.trainable_weights):,}")
print(f"Non-trainable parameters: {sum(p.numpy().size for p in model.non_trainable_weights):,}")

# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    'model_terbaik.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

history_phase2 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,
    callbacks=[early_stop, checkpoint, reduce_lr],
    verbose=1
)

print("\nFase 2 selesai!")
print(f"Training Accuracy: {history_phase2.history['accuracy'][-1]:.4f}")
print(f"Validation Accuracy: {history_phase2.history['val_accuracy'][-1]:.4f}")

## Plot Training History

In [ ]:
# Gabungkan history dari kedua fase
acc = history_phase1.history['accuracy'] + history_phase2.history['accuracy']
val_acc = history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy']
loss = history_phase1.history['loss'] + history_phase2.history['loss']
val_loss = history_phase1.history['val_loss'] + history_phase2.history['val_loss']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot Accuracy
ax1.plot(acc, label='Training Accuracy', linewidth=2)
ax1.plot(val_acc, label='Validation Accuracy', linewidth=2)
ax1.axvline(x=len(history_phase1.history['accuracy'])-1, color='red',
            linestyle='--', alpha=0.5, label='Fine-tune dimulai')
ax1.set_title('Accuracy', fontsize=14)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot Loss
ax2.plot(loss, label='Training Loss', linewidth=2)
ax2.plot(val_loss, label='Validation Loss', linewidth=2)
ax2.axvline(x=len(history_phase1.history['loss'])-1, color='red',
            linestyle='--', alpha=0.5, label='Fine-tune dimulai')
ax2.set_title('Loss', fontsize=14)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Training History (Fase 1 + Fase 2)', fontsize=16)
plt.tight_layout()
plt.show()

## Evaluasi: Confusion Matrix
Melihat kelas mana yang masih saling rancu.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Prediksi pada validation set
val_generator.reset()
y_pred = model.predict(val_generator, verbose=1)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = val_generator.classes

# Label kelas
class_labels = list(val_generator.class_indices.keys())

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_labels,
    yticklabels=class_labels
)
plt.title('Confusion Matrix', fontsize=14)
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Classification Report
print("\nClassification Report:")
print("=" * 60)
print(classification_report(y_true, y_pred_classes, target_names=class_labels))

## Simpan Model

In [ ]:
# Simpan model Keras
model.save("model_bangun_ruang_6kelas.keras")
print("Model Keras berhasil disimpan")

# Konversi ke ONNX
!pip install -U tf2onnx -q
import tf2onnx
import onnx

spec = (
    tf.TensorSpec(
        (None, 224, 224, 3),
        tf.float32,
        name="input"
    ),
)

output_path = "model_bangun_ruang_6kelas.onnx"

model_proto, _ = tf2onnx.convert.from_keras(
    model,
    input_signature=spec,
    opset=13,
    output_path=output_path
)

print(f"ONNX berhasil disimpan: {output_path}")

## Test Prediksi pada Gambar Baru
Uji model dengan gambar baru untuk memastikan prediksi benar.

In [ ]:
def predict_image(model, img_path, class_labels):
    """
    Prediksi satu gambar dan tampilkan hasilnya.
    Preprocessing: Edge Detection (sama seperti training)
    """
    # Baca dan resize gambar
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (224, 224))

    # Apply edge detection preprocessing
    processed = edge_detection_preprocessing(img_resized)

    # Prediksi
    input_batch = np.expand_dims(processed, axis=0)
    predictions = model.predict(input_batch, verbose=0)
    predicted_class = np.argmax(predictions[0])
    confidence = predictions[0][predicted_class]

    # Visualisasi
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    axes[0].imshow(img_rgb)
    axes[0].set_title('Original')

    vis = ((processed + 1.0) * 127.5).astype(np.uint8)
    axes[1].imshow(vis[:,:,0], cmap='gray')
    axes[1].set_title('Grayscale')
    axes[2].imshow(vis[:,:,1], cmap='gray')
    axes[2].set_title('Canny Edge')
    axes[3].imshow(vis[:,:,2], cmap='gray')
    axes[3].set_title('Sobel')

    for ax in axes:
        ax.axis('off')

    plt.suptitle(
        f'Prediksi: {class_labels[predicted_class]} ({confidence:.1%})',
        fontsize=14, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()

    # Tampilkan semua probabilitas
    print("\nProbabilitas per kelas:")
    for i, label in enumerate(class_labels):
        bar = '█' * int(predictions[0][i] * 30)
        print(f"  {label:20s} {predictions[0][i]:6.2%} {bar}")

# Contoh penggunaan:
# predict_image(model, '/path/to/gambar.jpg', class_labels)

# Test dengan beberapa gambar dari validasi
val_generator.reset()
for cls_name in class_labels:
    cls_path = os.path.join(dataset_path, cls_name)
    img_files = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg','.jpeg','.png','.jfif'))]
    if img_files:
        test_img = os.path.join(cls_path, img_files[-1])  # Ambil gambar terakhir
        print(f"\n{'='*50}")
        print(f"Label sebenarnya: {cls_name}")
        predict_image(model, test_img, class_labels)